In [7]:
import random

PRIME = 2**521 - 1  # A Mersenne prime; large enough for our purposes

def shamir_share(secret, n, t, prime=PRIME):
    """
    Split secret into n shares, any t of which reconstruct it.
    Returns a list of (x, y) tuples.
    """
    coeffs = [secret] + [random.randrange(1, prime) for _ in range(t - 1)]
    shares = []
    for i in range(1, n + 1):
        y = sum(c * pow(i, j, prime) for j, c in enumerate(coeffs)) % prime
        shares.append((i, y))
    return shares

def shamir_reconstruct(shares, prime=PRIME):
    """Reconstruct the secret from t shares using Lagrange interpolation."""
    secret = 0
    for j, (xj, yj) in enumerate(shares):
        num, den = 1, 1
        for m, (xm, _) in enumerate(shares):
            if m == j: continue
            num = (num * (-xm)) % prime
            den = (den * (xj - xm)) % prime
        lagrange = (num * pow(den, -1, prime)) % prime
        secret = (secret + yj * lagrange) % prime
    return secret

In [8]:
secret = 314159265
shares = shamir_share(secret, n=5, t=3)

# Any 3 shares should reconstruct the secret
import itertools
for combo in itertools.combinations(shares, 3):
    recovered = shamir_reconstruct(list(combo))
    assert recovered == secret, f"Failed for shares {[s[0] for s in combo]}"

print(f"Secret: {secret}")
print(f"All 3-of-5 combinations correctly reconstruct the secret.")

# But 2 shares should NOT
for combo in itertools.combinations(shares, 2):
    recovered = shamir_reconstruct(list(combo))
    # This will produce a wrong value, not the secret
    # (We don't assert != because of negligible collision probability)

Secret: 314159265
All 3-of-5 combinations correctly reconstruct the secret.


In [9]:
import random

p = 23  # Small prime for demonstration
g = 5   # Generator

# Prover's secret
x = 6
y = pow(g, x, p)

def schnorr_prover_commit():
    r = random.randrange(1, p)
    commitment = pow(g, r, p)
    return r, commitment

def schnorr_verifier_challenge():
    return random.choice([0, 1])  # Binary challenge for simplicity

def schnorr_prover_response(r, x, challenge):
    return (r + challenge * x) % (p - 1)

def schnorr_verify(y, commitment, challenge, response, g, p):
    left = pow(g, response, p)
    right = (commitment * pow(y, challenge, p)) % p
    return left == right

# Run one round
r, commitment = schnorr_prover_commit()
challenge = schnorr_verifier_challenge()
response = schnorr_prover_response(r, x, challenge)
valid = schnorr_verify(y, commitment, challenge, response, g, p)

print(f"Commitment: {commitment}")
print(f"Challenge:  {challenge}")
print(f"Response:   {response}")
print(f"Valid:      {valid}")

Commitment: 17
Challenge:  0
Response:   7
Valid:      True


 Cheating in a Single Round Can a cheating prover cheat? Yes, a prover who does not know the secret $x$ can cheat if they are able to predict the verifier's random challenge in advance1. If the cheating prover correctly guesses which challenge $e \in \{0, 1\}$ the verifier will choose, they can craft a false commitment $T$ that satisfies the verification equation for that specific challenge without knowing $x$12.Probability of success: Because interactive soundness relies entirely on the prover's inability to predict the random challenge, a cheating prover guessing a binary challenge ($0$ or $1$) has a $\frac{1}{2}$ (or 50%) probability of cheating successfully in a single round13.(b) Probability of Cheating Across 20 RoundsWhen the interaction is repeated for $20$ independent rounds, the verifier issues a fresh, unpredictable challenge in each round13.The probability that a cheating prover successfully guesses all $20$ challenges consecutively is: $$\left(\frac{1}{2}\right)^{20} = \frac{1}{1,048,576} \approx 9.537 \times 10^{-7} \text{ (or } \approx 0.00009537\% \text{)}$$Multiplying independent rounds reduces the cheating probability down to a negligible level, satisfying the protocol's formal soundness guarantee34.(c) The Zero-Knowledge PropertyWhat the verifier learns: Under the zero-knowledge property, the verifier learns nothing beyond the truth of the statement (i.e., that the prover possesses the secret $x$)4. The verifier gains zero information regarding the actual value of $x$45.Why it holds: The protocol transcript reveals no witness knowledge because a simulator can construct an algebraically indistinguishable transcript of commitments, challenges, and responses without ever knowing $x$4